# 06 V2: 30% volatility target with BSV

Input: V1 weights from 05 (`risk_adjusted_weights.csv`). Each day the 60-day covariance gives the forecast volatility of the V1 portfolio. If it is above 30%, all risk assets are scaled down by 30% / forecast and the released weight goes to BSV. Output: `results/signals/v2_weights_history.csv` (V2), used by 07 and 08.

Prices are loaded with `00_engine.load()` (US sessions only before returns). Fee rates and IS/OOS dates come from 00. Later cells test target levels of 30-60%.


**Which numbers to quote:** the tables in this notebook are research diagnostics. Quote final-portfolio performance from `results/tables/final_metrics.csv` (made by 07, also shown in `results/Report.html`).


In [ ]:
# ============================================================
# V2 - Step 1: Load inputs
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Project root
ROOT = Path.cwd().resolve()
while not (ROOT / "pipeline" / "00_engine.ipynb").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent   # works from the project root or any subfolder

# Shared calendar, fee table and period dates from 00_engine
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from portfolio_accounting import ENGINE
IS_START = ENGINE['IS_START']


print("Project root:", ROOT)

# Traded ETFs
ETFS = ["TQQQ", "UVXY", "TECL", "SPXL", "TECS", "BSV"]

# Main parameters
LOOKBACK = 60
ALPHA = 1.0
TARGET_VOL = 0.30
TRADING_DAYS = 252

# V1 daily weights from 05
weights_path = (
    ROOT / "results/signals/risk_adjusted_weights.csv"
)

v1_weights = pd.read_csv(
    weights_path,
    index_col=0,
    parse_dates=True
)[ETFS]

# V0 weights from 02 (for comparison with V2)
raw_path = (
    ROOT / "results/signals/ensemble_weights_history.csv"
)

raw_weights = pd.read_csv(
    raw_path,
    index_col=0,
    parse_dates=True
)[ETFS]

# ETF prices (US sessions only)
price_path = ROOT / "data/raw/etf_daily.csv"

prices = ENGINE['load'](price_path)  # Filter US sessions BEFORE returns.

# Use adjusted closing prices to compute returns
adj_close = prices["Adj Close"][ETFS]

daily_returns = adj_close.pct_change(fill_method=None)

# Keep dates where all six ETF returns are available
valid_returns = daily_returns[ETFS].dropna()

# Data checks
assert v1_weights.index.equals(raw_weights.index)
assert np.allclose(v1_weights.sum(axis=1), 1.0)
assert v1_weights.notna().all().all()

print("\n========== Data checks ==========")
print("Original ensemble weights: ", raw_weights.shape)
print("V1 risk-adjusted weights: ", v1_weights.shape)
print("ETF historical returns: ", valid_returns.shape)

print("\n========== Backtest dates ==========")
print("Start date: ", v1_weights.index.min())
print("End date: ", v1_weights.index.max())

print("\n========== V1 Average weights ==========")
display(
    (v1_weights.mean() * 100)
    .round(2)
    .to_frame("Average Weight (%)")
)

print("\n========== V1 Latest weights ==========")
display(
    (v1_weights.tail(1).T * 100)
    .round(2)
)

In [ ]:
# ============================================================
# V2 - Step 2: Daily forecast volatility of the V1 portfolio
# ============================================================

portfolio_vol = pd.Series(
    index=v1_weights.index,
    dtype=float,
    name="Forecast Portfolio Volatility"
)

for date in v1_weights.index:

    # Last 60 trading days of returns up to this date
    past_returns = (
        valid_returns
        .loc[:date, ETFS]
        .tail(LOOKBACK)
    )

    assert len(past_returns) == LOOKBACK, (
        f"{date}: not enough history"
    )

    # 60-day covariance matrix
    cov_matrix = past_returns.cov().values

    # Same-day V1 weights
    w = v1_weights.loc[date, ETFS].values

    # Forecast annualized portfolio volatility
    portfolio_vol.loc[date] = np.sqrt(
        TRADING_DAYS * (w @ cov_matrix @ w)
    )


# ------------------------------------------------------------
# Check results
# ------------------------------------------------------------

assert portfolio_vol.notna().all()
assert (portfolio_vol > 0).all()

print("========== V1 forecast portfolio volatility ==========")

display(
    portfolio_vol.describe()
    .to_frame("Annualized Volatility")
    .round(4)
)

print("\n========== Last 5 trading days ==========")

display(
    (portfolio_vol.tail(5) * 100)
    .round(2)
    .to_frame("Forecast Vol (%)")
)

print("\n========== Share of days above the 30% target ==========")

above_target = (portfolio_vol > TARGET_VOL).mean()

print(f"{above_target:.2%}")

print("\n========== Latest forecast volatility ==========")

print(f"{portfolio_vol.iloc[-1]:.2%}")

In [ ]:
# ============================================================
# V2 - Step 3: Scale down risk assets, move the rest to BSV
# ============================================================

# 1. Calculate risk exposure scaling factors
risk_scale = (
    TARGET_VOL / portfolio_vol
).clip(upper=1.0)

# Check scaling factors
assert risk_scale.notna().all()
assert ((risk_scale > 0) & (risk_scale <= 1)).all()

# 2. Start from V1 weights
v2_weights = v1_weights.copy()

# 3. Scale only the non-BSV assets
RISK_ASSETS = [etf for etf in ETFS if etf != "BSV"]

v2_weights[RISK_ASSETS] = (
    v1_weights[RISK_ASSETS]
    .mul(risk_scale, axis=0)
)

# 4. Allocate all released capital to BSV
v2_weights["BSV"] = (
    1.0 - v2_weights[RISK_ASSETS].sum(axis=1)
)

# ------------------------------------------------------------
# 5. Data checks
# ------------------------------------------------------------

assert v2_weights.notna().all().all()

assert (v2_weights >= -1e-10).all().all()

assert np.allclose(
    v2_weights.sum(axis=1),
    1.0
), "Weights do not sum to 100% on some dates"

assert (
    v2_weights["BSV"] >= v1_weights["BSV"] - 1e-10
).all(), "BSV weight fell below V1 on some dates"

print("========== Average weights: V1 vs V2 ==========")

display(
    pd.DataFrame({
        "V1 (%)": v1_weights.mean() * 100,
        "V2 (%)": v2_weights.mean() * 100
    }).round(2)
)

# ------------------------------------------------------------
# 6. Latest weight comparison
# ------------------------------------------------------------

print("\n========== Latest trading day weights ==========")

latest_comparison = pd.DataFrame({
    "V1 (%)": v1_weights.iloc[-1] * 100,
    "V2 (%)": v2_weights.iloc[-1] * 100
})

display(latest_comparison.round(2))

# ------------------------------------------------------------
# 7. Defensive mechanism triggers
# ------------------------------------------------------------

print("\n========== BSV defence statistics ==========")

print(
    f"Share of days with risk scaling: "
    f"{(risk_scale < 1).mean():.2%}"
)

print(
    f"Average risk scale factor: "
    f"{risk_scale.mean():.2%}"
)

print(
    f"V1 average BSV weight: "
    f"{v1_weights['BSV'].mean():.2%}"
)

print(
    f"V2 average BSV weight: "
    f"{v2_weights['BSV'].mean():.2%}"
)

print(
    f"V2 maximum BSV weight: "
    f"{v2_weights['BSV'].max():.2%}"
)

In [ ]:
# ============================================================
# V2 - Step 4: Forecast volatility before and after adding BSV
# ============================================================

# 1. V2 daily forecast volatility
v2_portfolio_vol = pd.Series(
    index=v2_weights.index,
    dtype=float,
    name="V2 Forecast Volatility"
)

for date in v2_weights.index:

    # Last 60 trading days of returns up to that close
    past_returns = (
        valid_returns
        .loc[:date, ETFS]
        .tail(LOOKBACK)
    )

    # Calculate covariance matrix
    cov_matrix = past_returns.cov().values

    # V2 weights
    w = v2_weights.loc[date, ETFS].values

    # Calculate forecast annualized portfolio volatility
    v2_portfolio_vol.loc[date] = np.sqrt(
        TRADING_DAYS * (w @ cov_matrix @ w)
    )


# ------------------------------------------------------------
# 2. Data checks
# ------------------------------------------------------------

assert v2_portfolio_vol.notna().all()
assert (v2_portfolio_vol >= 0).all()


# ------------------------------------------------------------
# 3. V1 vs V2 Volatility comparison
# ------------------------------------------------------------

vol_comparison = pd.DataFrame({
    "V1": portfolio_vol,
    "V2": v2_portfolio_vol
})

print("========== Forecast volatility: V1 vs V2 ==========")

display(
    (vol_comparison.describe() * 100)
    .round(2)
)


# ------------------------------------------------------------
# 4. Average volatility
# ------------------------------------------------------------

print("\n========== Average forecast annualized volatility ==========")

print(f"V1: {portfolio_vol.mean():.2%}")
print(f"V2: {v2_portfolio_vol.mean():.2%}")

print(
    f"Reduction: "
    f"{(portfolio_vol.mean() - v2_portfolio_vol.mean()):.2%}"
)


# ------------------------------------------------------------
# 5. Share of days above the 30% target
# ------------------------------------------------------------

print("\n========== Share of days above the 30% target ==========")

print(
    f"V1: {(portfolio_vol > TARGET_VOL).mean():.2%}"
)

print(
    f"V2: {(v2_portfolio_vol > TARGET_VOL).mean():.2%}"
)


# ------------------------------------------------------------
# 6. Last 5 trading days
# ------------------------------------------------------------

print("\n========== Last 5 days of forecast volatility ==========")

display(
    (vol_comparison.tail(5) * 100)
    .round(2)
)


# ------------------------------------------------------------
# 7. Latest trading day
# ------------------------------------------------------------

print("\n========== Latest trading day ==========")

print(f"V1: {portfolio_vol.iloc[-1]:.2%}")
print(f"V2: {v2_portfolio_vol.iloc[-1]:.2%}")


# ------------------------------------------------------------
# 8. Is the forecast volatility lower every day?
# ------------------------------------------------------------

lower_vol_days = (
    v2_portfolio_vol < portfolio_vol - 1e-10
).mean()

print("\n========== Proportion of trading days with lower risk ==========")

print(f"{lower_vol_days:.2%}")

In [ ]:
# ============================================================
# V2 - Step 5: V0 / V1 / V2 Backtest performance comparison
# ============================================================

import json
import ast

# ------------------------------------------------------------
# 1. Load backtest_weighted_portfolio() from 02
# ------------------------------------------------------------

# Shared exact transaction-cost solver used by the extracted portfolio function.
_engine_ns = ENGINE
rebalance_exact = ENGINE['rebalance_exact']

def load_function(notebook_path, function_name):

    with open(notebook_path, encoding="utf-8") as f:
        notebook = json.load(f)

    for cell in notebook["cells"]:

        if cell["cell_type"] != "code":
            continue

        source = "".join(cell["source"])

        try:
            tree = ast.parse(source)
        except SyntaxError:
            continue

        for node in tree.body:

            if (
                isinstance(node, ast.FunctionDef)
                and node.name == function_name
            ):

                module = ast.Module(
                    body=[node],
                    type_ignores=[]
                )

                exec(
                    compile(module, "<notebook>", "exec"),
                    globals()
                )

                return

    raise ValueError(f"Cannot find function: {function_name}")


load_function(
    next((ROOT / "pipeline").rglob("02_parameter_ensemble.ipynb")),
    "backtest_weighted_portfolio"
)

metrics = ENGINE["metrics"]   # shared performance metrics from 00_engine

print("Backtest and performance functions loaded")


# ------------------------------------------------------------
# 2. One-way costs from 00_engine
# ------------------------------------------------------------

TICKS = ETFS

ONE_WAY_BPS = {t: ENGINE['ONE_WAY_BPS'][t] for t in ETFS}


# ------------------------------------------------------------
# 3. Adjusted opening and closing prices (same basis as 05)
# ------------------------------------------------------------

price_data = prices.sort_index()

adjustment_factor = (
    price_data["Adj Close"][ETFS]
    / price_data["Close"][ETFS]
)

open_adjusted = (
    price_data["Open"][ETFS]
    * adjustment_factor
)

O = open_adjusted.reindex(raw_weights.index)[ETFS]

C = (
    price_data["Adj Close"][ETFS]
    .reindex(raw_weights.index)
)

assert O.notna().all().all()
assert C.notna().all().all()


# ------------------------------------------------------------
# 4. Backtest three strategies
# ------------------------------------------------------------

strategies = {
    "V0 Original": raw_weights,
    "V1 Inverse Vol": v1_weights,
    "V2 BSV Targeting": v2_weights
}

equity_results = {}
diagnostics = {}

for name, weights in strategies.items():

    equity, diag = backtest_weighted_portfolio(
        O,
        C,
        weights,
        start=IS_START
    )

    equity_results[name] = equity
    diagnostics[name] = diag

    print(
        f"{name}: "
        f"Final Equity = {equity.iloc[-1]:.4f}"
    )


# ------------------------------------------------------------
# 5. Check that V0 / V1 reproduce the 05 backtest, day by day on common dates,
#    against the daily equity that 05 saved in risk_equity_history.csv
# ------------------------------------------------------------

ref_file = ROOT / "results" / "tables" / "risk_equity_history.csv"

if ref_file.exists():
    ref = pd.read_csv(ref_file, index_col="Date", parse_dates=True)
    for name, col in [("V0 Original", "Original"), ("V1 Inverse Vol", "Risk_Adjusted")]:
        common = equity_results[name].index.intersection(ref.index)
        assert len(common) > 0, f"{name}: no dates in common with the 05 equity series"
        assert np.allclose(
            equity_results[name].loc[common].values,
            ref[col].loc[common].values,
            rtol=1e-5
        ), f"{name} does not match 05: rerun 05 on the same data, then this notebook"
    print(f"\nV0 and V1 match the 05 backtest ({len(common)} common dates)")
else:
    print("\n⚠ risk_equity_history.csv from 05 not found; skipping the V0/V1 check (run 05 first)")


# ------------------------------------------------------------
# 6. Backtest periods
# ------------------------------------------------------------

periods = {
    "IS": (ENGINE["IS_START"], ENGINE["IS_END"]),
    "OOS": (ENGINE["OOS_START"], ENGINE["OOS_END"]),
    "OOS ex-2020": ("2021-01-01", ENGINE["OOS_END"])
}


# ------------------------------------------------------------
# 7. Performance metrics
# ------------------------------------------------------------

results = []

for name, equity in equity_results.items():

    for period, (start, end) in periods.items():

        performance = metrics(
            equity,
            a=start,
            b=end
        )

        results.append({
            "Strategy": name,
            "Period": period,
            **performance
        })


v2_performance = pd.DataFrame(results)


# ------------------------------------------------------------
# 8. Show each period
# ------------------------------------------------------------

for period in periods:

    print(f"\n========== {period} ==========")

    display(
        v2_performance[
            v2_performance["Period"] == period
        ].round(4)
    )


In [ ]:
# ============================================================
# V2 - Step 6: Target volatility sensitivity tests
# ============================================================

TARGET_VOLS = [0.30, 0.40, 0.50, 0.60]

target_results = []
target_weights = {}
target_equities = {}

RISK_ASSETS = [etf for etf in ETFS if etf != "BSV"]

# ------------------------------------------------------------
# 1. Test different target volatilities
# ------------------------------------------------------------

for target_vol in TARGET_VOLS:

    print(f"Testing target volatility: {target_vol:.0%}")

    # Scale factor from the V1 forecast volatility
    scale = (target_vol / portfolio_vol).clip(upper=1.0)

    # Start from V1 weights
    weights = v1_weights.copy()

    # Scale the non-BSV assets
    weights[RISK_ASSETS] = (
        v1_weights[RISK_ASSETS]
        .mul(scale, axis=0)
    )

    # Allocate all remaining capital to BSV
    weights["BSV"] = (
        1.0 - weights[RISK_ASSETS].sum(axis=1)
    )

    # --------------------------------------------------------
    # 2. Check weights
    # --------------------------------------------------------

    assert weights.notna().all().all()

    assert (weights >= -1e-10).all().all()

    assert np.allclose(
        weights.sum(axis=1),
        1.0
    )

    # Save weights for each target
    target_weights[target_vol] = weights

    # --------------------------------------------------------
    # 3. Backtest with the 02 portfolio function
    # --------------------------------------------------------

    equity, diag = backtest_weighted_portfolio(
        O,
        C,
        weights,
        start=IS_START
    )

    target_equities[target_vol] = equity

    # --------------------------------------------------------
    # 4. Record performance by period
    # --------------------------------------------------------

    for period, (start, end) in periods.items():

        performance = metrics(
            equity,
            a=start,
            b=end
        )

        target_results.append({
            "Target Vol": target_vol,
            "Period": period,
            "Average BSV": weights["BSV"].mean(),
            "Average TQQQ": weights["TQQQ"].mean(),
            **performance
        })

    print(f"Target volatility {target_vol:.0%} done")


# ------------------------------------------------------------
# 5. Summarize results
# ------------------------------------------------------------

target_comparison = pd.DataFrame(target_results)

print("\n========== All test results ==========")

display(
    target_comparison.round(4)
)


# ------------------------------------------------------------
# 6. Show IS / OOS / OOS ex-2020 separately
# ------------------------------------------------------------

for period in periods:

    print(f"\n========== {period} ==========")

    display(
        target_comparison[
            target_comparison["Period"] == period
        ].round(4)
    )


# ------------------------------------------------------------
# 7. Check that the 30% target reproduces V2
# ------------------------------------------------------------

print("\n========== 30% target consistency check ==========")

same_weights = np.allclose(
    target_weights[0.30][ETFS].values,
    v2_weights[ETFS].values
)

print("30% target weights match Step 3: ", same_weights)

assert same_weights, "30% target did not reproduce the V2 weights"

assert np.isclose(
    target_equities[0.30].iloc[-1],
    equity_results["V2 BSV Targeting"].iloc[-1],
    rtol=1e-8
), "30% target did not reproduce the V2 equity"

print("30% target equity check passed")

In [ ]:
# Latest date of each input and output

print("Last row of price CSV: ", prices.index.max())

print(
    "Last date with all six ETF prices: ",
    prices["Adj Close"][ETFS].dropna().index.max()
)

print("Last V0 date: ", raw_weights.index.max())

print("Last V1 date: ", v1_weights.index.max())

print("Last V2 date: ", v2_weights.index.max())

print(
    "Last V0 equity date: ",
    equity_results["V0 Original"].index.max()
)

In [ ]:
# ============================================================
# V2 - Step 7: Export V2 daily weights (target volatility TARGET_VOL)
# Read by 07_final_strategy (historical backtest) and 08_final_strategy_live (live tracking)
# ============================================================

SIGNAL_DIR = ROOT / "results" / "signals"
SIGNAL_DIR.mkdir(parents=True, exist_ok=True)

v2_out = v2_weights[ETFS].copy()      # V2 weights from Step 3 (target volatility = TARGET_VOL)
v2_out.index.name = "Date"

assert np.allclose(v2_out.sum(axis=1), 1.0)
assert (v2_out >= -1e-10).all().all()

v2_out.to_csv(SIGNAL_DIR / "v2_weights_history.csv", encoding="utf-8-sig")

print("Saved V2 daily weights: ", SIGNAL_DIR / "v2_weights_history.csv")
print(f"Target volatility: {TARGET_VOL:.0%} | Period: {v2_out.index.min().date()} ~ {v2_out.index.max().date()}")
print("\nLatest daily weights (%): "); display((v2_out.iloc[-1] * 100).round(2).to_frame("Weight (%)"))
